In [ ]:
# -------------------------------
# Bronze Layer Ingestion - Data
# -------------------------------
# Purpose:
# - Ingest raw CSV files from Files (landing zone)
# - Add file-level metadata for traceability
# - Load only NEW files using checkpointing
# - Store data in Bronze Delta table
# - Track processed files in checkpoint table
# ------------------------------------------------

from pyspark.sql.functions import current_timestamp, col

folders = ["diagnosis", "hospital", "patient", "visit"]

for folder in folders:
    # ---------------------------------------
    # Step 1: Read raw data from landing zone
    # ---------------------------------------
    raw_df = (
        spark.read.format("csv")              # Source format (can be json/parquet if needed)
        .option("header", True)               # First row contains column names
        .option("mergeSchema", True)
        .load(f"Files/healthcare_data/{folder}/")
    )

    # -----------------------------------------------------------------
    # Step 2: Add file metadata (important for deduplication & lineage)
    # -----------------------------------------------------------------
    df_with_file = raw_df.withColumn(
        "file_name",
        col("_metadata.file_name")            # Extract file name from metadata
    )

    # -------------------------------------------------------
    # Step 3: Load checkpoint table (already processed files)
    # -------------------------------------------------------
    processed_files_df = spark.read.table(f"DE_Learning_LH.{folder}_bronze.checkpoint_table")

    # -----------------------------------------------------
    # Step 4: Filter only NEW files (incremental ingestion)
    # -----------------------------------------------------
    new_data_df = df_with_file.join(
        processed_files_df,
        on="file_name",
        how="left_anti"                       # Keep only records NOT in checkpoint
    )
 
    if not new_data_df.isEmpty():
        # --------------------------------------
        # Step 5: Write new data to Bronze table
        # --------------------------------------
        (
            new_data_df
            .drop("file_name")                   # Optional: remove metadata column before storing
            .write
            .format("delta")                     # Always use Delta in Fabric for reliability
            .mode("append")                      # Append new data
            .option("mergeSchema", "true")       # Allow schema evolution
            .saveAsTable(f"DE_Learning_LH.{folder}_bronze.{folder}")
        )

    # --------------------------------------------
    # Step 6: Capture metadata for processed files
    # --------------------------------------------
    new_files_df = (
        df_with_file
        .select(
            col("_metadata.file_path").alias("file_path"),
            col("_metadata.file_name").alias("file_name"),
            col("_metadata.file_size").cast("double").alias("file_size")
        )
        .distinct()                          # Avoid duplicates
        .withColumn("processed_ts", current_timestamp())
    )

    # -------------------------------
    # Step 7: Update checkpoint table
    # -------------------------------
    (
        new_files_df
        .write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(f"DE_Learning_LH.{folder}_bronze.checkpoint_table")
    )